# Subgroups

##### Sections
- [What are Subgroups?](#What-are-Subgroups?)
- [How a Subgroup Maps to Graphics Hardware](#How-a-Subgroup-Maps-to-Graphics-Hardware)
- _Code:_ [Subgroup info](#Subgroup-info)
- _Code:_ [Subgroup Size](#Subgroup-Size)
- [Subgroup Functions and Algorithms](#Subgroup-Functions-and-Algorithms)
- _Code:_ [Subgroup Shuffle](#Subgroup-Shuffle)
- _Code:_ [Subgroup - Reduce](#Code-Example:-Subgroup---Reduce)
- _Code:_ [Subgroup - Broadcast](#Code-Example:-Subgroup---Broadcast)
- _Code:_ [Subgroup - Votes](#Code-Example:-Subgroup---Votes)
- _Lab Exercise:_ [Sub-Groups](#Lab-Exercise:-Sub-Groups)

## Learning Objectives

- Understand advantages of using Subgroups in SYCL
- Take advantage of Subgroup algorithms for performance and productivity
- Use Subgroup Shuffle operations to avoid explicit memory operations

## What are Subgroups?

On many modern hardware platforms, __a subset of the work-items in a work-group__ are executed simultaneously or with additional scheduling guarantees. These subset of work-items are called subgroups. Leveraging subgroups will help to __map execution to low-level hardware__ and may help in achieving higher performance.

## Subgroups in ND-Range Kernel Execution

Parallel execution with the ND_RANGE Kernel helps to group work items that map to hardware resources. This helps to __tune applications for performance__.

The execution range of an ND-range kernel is divided into __work-groups__, __subgroups__ and __work-items__ as shown in picture below.

![ND-range kernel execution](assets/ndrange.png)

## How a Subgroup Maps to Graphics Hardware

| | |
|:---:|:---|
| __Work-item__ | Represents the individual instances of a kernel function. | 
| __Work-group__ | The entire iteration space is divided into smaller groups called work-groups, work-items within a work-group are scheduled on a single compute unit on hardware. | 
| __Subgroup__ | A subset of work-items within a work-group that are executed simultaneously, may be mapped to vector hardware. | 


The picture below shows how work-groups and subgroups map to __Intel® Gen11 Graphics Hardware__.

![ND-Range Hardware Mapping](assets/hwmapping.png)

## Why use Subgroups?

- Work-items in a sub-group can __communicate directly using shuffle operations__, without explicit memory operations.
- Work-items in a sub-group can synchronize using sub-group barriers and __guarantee memory consistency__ using sub-group memory fences.
- Work-items in a sub-group have access to __sub-group functions and algorithms__, providing fast implementations of common parallel patterns.

## sub_group class

The subgroup handle can be obtained from the nd_item using the __get_sub_group()__

```cpp
        sycl::sub_group sg = nd_item.get_sub_group();

                 OR

        auto sg = nd_item.get_sub_group();
```

Once you have the subgroup handle, you can query for more information about the subgroup, do shuffle operations or use group algorithm.

## Subgroup info

The subgroup handle can be queried to get other information like number of work-items in subgroup, or number of subgroups in a work-group which will be needed for developers to implement kernel code using subgroups:
- __get_local_id()__ returns the index of the work-item within its subgroup
- __get_local_range()__ returns the size of sub_group 
- __get_group_id()__ returns the index of the subgroup
- __get_group_range()__ returns the number of subgroups within the parent work-group


```cpp
    h.parallel_for(nd_range<1>(64,64), [=](nd_item<1> item){
      /* get sub_group handle */
      auto sg = item.get_sub_group();
      /* query sub_group and print sub_group info once per sub_group */
      if(sg.get_local_id()[0] == 0){
        out << "sub_group id: " << sg.get_group_id()[0]
            << " of " << sg.get_group_range()[0]
            << ", size=" << sg.get_local_range()[0] 
            << "\n";
      }
    });
```

### Code Example: Subgroup Info

The SYCL code below demonstrates subgroup query methods to print sub-group info: Inspect code, there are no modifications necessary:
1. Inspect the code cell below and click run ▶ to save the code to file
2. Next run ▶ the cell in the __Build and Run__ section below the code to compile and execute the code.

In [67]:
%%writefile lab/sub_group_info.cpp
//==============================================================
// Copyright © Intel Corporation
//
// SPDX-License-Identifier: MIT
// =============================================================
#include <sycl/sycl.hpp>
using namespace sycl;

static constexpr size_t N = 64; // global size
static constexpr size_t B = 64; // work-group size

int main() {
  queue q;
  std::cout << "Device : " << q.get_device().get_info<info::device::name>() << "\n";

  q.submit([&](handler &h) {
    //# setup sycl stream class to print standard output from device code
    auto out = stream(1024, 768, h);

    //# nd-range kernel
    h.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item) {
      //# get sub_group handle
      auto sg = item.get_sub_group();

      //# query sub_group and print sub_group info once per sub_group
      if (sg.get_local_id()[0] == 0) {
        out << "sub_group id: " << sg.get_group_id()[0] << " of "
            << sg.get_group_range()[0] << ", size=" << sg.get_local_range()[0]
            << "\n";
      }
    });
  }).wait();
}

Overwriting lab/sub_group_info.cpp


#### Build and Run
Select the cell below and click run ▶ to compile and execute the code:

In [68]:
! chmod 755 q; chmod 755 run_sub_group_info.sh; if [ -x "$(command -v qsub)" ]; then ./q run_sub_group_info.sh; else ./run_sub_group_info.sh; fi

## u3eed2aeb1316a4735ff54db21771fda is compiling SYCL_Essentials Module4 -- SYCL Sub Groups - 1 of 7 sub_group_info.cpp
Device : Intel(R) Data Center GPU Max 1100
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 0 of 2, size=32
sub_group id: 0 of 2, size=

## Subgroup Size

For tuning applications for performance, sub-group size may have to be set a specific value. For example, Intel(R) GPU supports sub-groups sizes of 8, 16 and 32; by default the compiler implementation will pick optimal sub-group size, but it can also be forced to use a specific value.

The supported sub-group sizes for a GPU can be queried from device information as shown below:

```cpp
auto sg_sizes = q.get_device().get_info<info::device::sub_group_sizes>();
                                                      ^
```

`reqd_sub_group_size(S)` allows setting a specific sub-group size to use for kernel execution, the specified value should be one of the supported sizes and must be a compile time constant value.

```cpp
    q.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item)[[intel::reqd_sub_group_size(16)]] {
                                                          ^
        // Kernel Code
        
    }).wait();

```

### Code Example: Subgroup Size

The code below shows how to query for supported sub-group sizes, and also how to set kernel to use a specific supported sub-group size.

The SYCL code below demonstrates how to use reqd_sub_group_size() to let the kernel use a specified sub-group size, change the __`S = 32`__ to __16__ or __8__ to change sub_group sizes and check the output:

1. Inspect the code cell below and click run ▶ to save the code to file.

2. Next run ▶ the cell in the __Build and Run__ section below the code to compile and execute the code.

In [11]:
%%writefile lab/sub_group_reqd_size.cpp
//==============================================================
// Copyright © Intel Corporation
//
// SPDX-License-Identifier: MIT
// =============================================================
#include <sycl/sycl.hpp>
using namespace sycl;

static constexpr size_t N = 64; // global size
static constexpr size_t B = 64; // work-group size

int main() {
  queue q;
  std::cout << "Device : " << q.get_device().get_info<info::device::name>() << "\n";

  //# get all supported sub_group sizes and print
  auto sg_sizes = q.get_device().get_info<info::device::sub_group_sizes>();
  std::cout << "Supported Sub-Group Sizes : ";
  for (int i=0; i<sg_sizes.size(); i++) std::cout << sg_sizes[i] << " "; std::cout << "\n";
    
  //# find out maximum supported sub_group size
  auto max_sg_size = std::max_element(sg_sizes.begin(), sg_sizes.end());
  std::cout << "Max Sub-Group Size        : " << max_sg_size[0] << "\n";
    
  q.submit([&](handler &h) {
    //# setup sycl stream class to print standard output from device code
    auto out = stream(1024, 768, h);

    //# nd-range kernel with user specified sub_group size
    h.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item)[[intel::reqd_sub_group_size(32)]] {
      //# get sub_group handle
      auto sg = item.get_sub_group();

      //# query sub_group and print sub_group info once per sub_group
      if (sg.get_local_id()[0] == 0) {
        out << "sub_group id: " << sg.get_group_id()[0] << " of "
            << sg.get_group_range()[0] << ", size=" << sg.get_local_range()[0]
            << "\n";
      }
    });
  }).wait();
}

Overwriting lab/sub_group_reqd_size.cpp


#### Build and Run
Select the cell below and click run ▶ to compile and execute the code:

In [12]:
! chmod 755 q; chmod 755 run_sub_group_reqd_size.sh; if [ -x "$(command -v qsub)" ]; then ./q run_sub_group_reqd_size.sh; else ./run_sub_group_reqd_size.sh; fi

## u3eed2aeb1316a4735ff54db21771fda is compiling SYCL_Essentials Module4 -- SYCL Sub Groups - 2 of 7 sub_group_reqd_size.cpp
Device : Intel(R) Data Center GPU Max 1100
Supported Sub-Group Sizes : 16 32 
Max Sub-Group Size        : 32
sub_group id: 1 of 2, size=32
sub_group id: 0 of 2, size=32


## Subgroup Functions and Algorithms

The sub-group functions and algorithms expose functionality tied to work-items within a sub-group.  

Providing these implementations as library functions instead __increases developer productivity__ and gives implementations the ability to __generate highly optimized 
code__ for individual target devices.

Below are some of the group algorithms available for sub-groups, they include useful functionalities to perform shuffles, reductions, scans and votes:

- select_by_group
- shift_group_left
- shift_group_right
- permute_group_by_xor
- group_broadcast
- reduce_over_group
- exclusive_scan_over_group
- inclusive_scan_over_group
- any_of_group
- all_of_group
- none_of_group

## Subgroup Shuffle

One of the most useful features of subgroups is the ability to __communicate directly between individual work-items__ without explicit memory operations.

Shuffle operations enable us to remove work-group local memory usage from our kernels and/or to __avoid unnecessary repeated accesses to global memory__.

Below are the different types of shuffle operations available for sub-groups:
- `select_by_group(sg, x, id)`
- `shift_group_left(sg, x, delta)`
- `shift_group_right(sg, x, delta)`
- `permute_group_by_xor(sg, x, mask)`

The code below uses `permute_group_by_xor` to swap the values of two work-items:

```cpp
    h.parallel_for(nd_range<1>(N,B), [=](nd_item<1> item){
      auto sg = item.get_sub_group();
      auto i = item.get_global_id(0);
      /* Shuffles */
      //data[i] = select_by_group(sg, data[i], 2);
      //data[i] = shift_group_left(sg, data[i], 1);
      //data[i] = shift_group_right(sg, data[i], 1);
      data[i] = permute_group_by_xor(sg, data[i], 1);
    });

```

<img src="assets/shuffle_xor.png" alt="shuffle_xor" width="300"/>

### Code Example: Subgroup Shuffle

The code below uses subgroup shuffle to swap items in a subgroup. You can try other shuffle operations or change the fixed constant in the shuffle function to express some common communication patterns using `permute_group_by_xor`.

The SYCL code below demonstrates sub-group shuffle operations, the code shows how `permute_group_by_xor` can be used to swap adjacent elements in sub-group, and also you can change the code to reverse the order of element in sub-group using a different mask.

1. Inspect the code cell below and click run ▶ to save the code to file.

2. Next run ▶ the cell in the __Build and Run__ section below the code to compile and execute the code.

In [25]:
%%writefile lab/sub_group_shuffle.cpp
//==============================================================
// Copyright © Intel Corporation
//
// SPDX-License-Identifier: MIT
// =============================================================
#include <sycl/sycl.hpp>
using namespace sycl;

static constexpr size_t N = 256; // global size
static constexpr size_t B = 64;  // work-group size

int main() {
  queue q;
  std::cout << "Device : " << q.get_device().get_info<info::device::name>() << "\n";

  //# initialize data array using usm
  int *data = malloc_shared<int>(N, q);
  for (int i = 0; i < N; i++) data[i] = i;
  for (int i = 0; i < N; i++) std::cout << data[i] << " ";
  std::cout << "\n\n";

  q.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item) {
    auto sg = item.get_sub_group();
    auto i = item.get_global_id(0);

    //# swap adjacent items in array using sub_group permute_group_by_xor
    //data[i] = permute_group_by_xor(sg, data[i], 1);
      
    //# reverse the order of items in sub_group using permute_group_by_xor
    data[i] = permute_group_by_xor(sg, data[i], sg.get_max_local_range()[0] - 1);
      
  }).wait();

  for (int i = 0; i < N; i++) std::cout << data[i] << " ";
  std::cout << "\n";

  free(data, q);
  return 0;
}

Overwriting lab/sub_group_shuffle.cpp


#### Build and Run
Select the cell below and click run ▶ to compile and execute the code:

In [26]:
! chmod 755 q; chmod 755 run_sub_group_shuffle.sh; if [ -x "$(command -v qsub)" ]; then ./q run_sub_group_shuffle.sh; else ./run_sub_group_shuffle.sh; fi

## u3eed2aeb1316a4735ff54db21771fda is compiling SYCL_Essentials Module4 -- SYCL Sub Groups - 3 of 7 sub_group_shuffle.cpp
Device : Intel(R) Data Center GPU Max 1100
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 

### Code Example: Subgroup - Reduce

The code below uses subgroup `reduce_over_group` function to perform reduction for all items in a subgroup. 

```cpp
    h.parallel_for(nd_range<1>(N,B), [=](nd_item<1> item){
      auto sg = item.get_sub_group();
      auto i = item.get_global_id(0);
      /* Reduction algorithm on Sub-group */
      int result = reduce_over_group(sg, data[i], plus<>());
      //int result = reduce_over_group(sg, data[i], maximum<>());
      //int result = reduce_over_group(sg, data[i], minimum<>());
    });

```

The SYCL code below demonstrates sub-group algorithm: Inspect code, you can change the operator "_plus_" to "_maximum_" or "_minimum_" and check output:

1. Inspect the code cell below and click run ▶ to save the code to file.

2. Next run ▶ the cell in the __Build and Run__ section below the code to compile and execute the code.

In [33]:
%%writefile lab/sub_group_reduce.cpp
//==============================================================
// Copyright © Intel Corporation
//
// SPDX-License-Identifier: MIT
// =============================================================
#include <sycl/sycl.hpp>
using namespace sycl;

static constexpr size_t N = 256; // global size
static constexpr size_t B = 64;  // work-group size

int main() {
  queue q;
  std::cout << "Device : " << q.get_device().get_info<info::device::name>() << "\n";

  //# initialize data array using usm
  int *data = malloc_shared<int>(N, q);
  for (int i = 0; i < N; i++) data[i] = i;
  for (int i = 0; i < N; i++) std::cout << data[i] << " ";
  std::cout << "\n\n";

  q.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item) {
    auto sg = item.get_sub_group();
    auto i = item.get_global_id(0);

    //# Add all elements in sub_group using sub_group algorithm
    int result = reduce_over_group(sg, data[i], plus<>());

    //# write sub_group sum in first location for each sub_group
    if (sg.get_local_id()[0] == 0) {
      data[i] = result;
    } else {
      data[i] = 0;
    }
  }).wait();

  for (int i = 0; i < N; i++) std::cout << data[i] << " ";
  std::cout << "\n";

  free(data, q);
  return 0;
}

Overwriting lab/sub_group_reduce.cpp


#### Build and Run
Select the cell below and click run ▶ to compile and execute the code:

In [34]:
! chmod 755 q; chmod 755 run_sub_group_reduce.sh; if [ -x "$(command -v qsub)" ]; then ./q run_sub_group_reduce.sh; else ./run_sub_group_reduce.sh; fi

## u3eed2aeb1316a4735ff54db21771fda is compiling SYCL_Essentials Module4 -- SYCL Sub Groups - 4 of 7 sub_group_reduce.cpp
Device : Intel(R) Data Center GPU Max 1100
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 2

### Code Example: Subgroup - Broadcast

The code below uses subgroup algorithm `group_broadcast` function, this enables one work-item in a group to share the value of a variable with all other work-items in the group.

The SYCL code below demonstrates sub-group broadcast function: Inspect code, there are no modifications necessary:

1. Inspect the code cell below and click run ▶ to save the code to file.

2. Next run ▶ the cell in the __Build and Run__ section below the code to compile and execute the code.

In [35]:
%%writefile lab/sub_group_broadcast.cpp
//==============================================================
// Copyright © Intel Corporation
//
// SPDX-License-Identifier: MIT
// =============================================================
#include <sycl/sycl.hpp>
using namespace sycl;

static constexpr size_t N = 256; // global size
static constexpr size_t B = 64; // work-group size

int main() {
  queue q;
  std::cout << "Device : " << q.get_device().get_info<info::device::name>() << "\n";

  //# initialize data array using usm
  int *data = malloc_shared<int>(N, q);
  for(int i=0; i<N; i++) data[i] = i;
  for(int i=0; i<N; i++) std::cout << data[i] << " "; 
  std::cout << "\n\n";  

  //# use parallel_for and sub_groups
  q.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item) {
    auto sg = item.get_sub_group();
    auto i = item.get_global_id(0);

    //# write sub_group item values to broadcast value at index 3
    data[i] = group_broadcast(sg, data[i], 3);

  }).wait();

  for(int i=0; i<N; i++) std::cout << data[i] << " "; 
  std::cout << "\n";
  
  free(data, q);
  return 0;
}

Writing lab/sub_group_broadcast.cpp


#### Build and Run
Select the cell below and click run ▶ to compile and execute the code:

In [36]:
! chmod 755 q; chmod 755 run_sub_group_broadcast.sh; if [ -x "$(command -v qsub)" ]; then ./q run_sub_group_broadcast.sh; else ./run_sub_group_broadcast.sh; fi

## u3eed2aeb1316a4735ff54db21771fda is compiling SYCL_Essentials Module4 -- SYCL Sub Groups - 5 of 7 sub_group_broadcast.cpp
Device : Intel(R) Data Center GPU Max 1100
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 23

### Code Example: Subgroup - Votes

The `any_of_group`, `all_of_group` and `none_of_group` functions (henceforth referred to collectively as
“vote” functions) enable work-items to compare the result of a Boolean
condition across their group.

The SYCL code below demonstrates sub-group algorithms `any_of_group`, `all_of_group` and `none_of_group` functions: Inspect code, there are no modifications necessary:

1. Inspect the code cell below and click run ▶ to save the code to file.

2. Next run ▶ the cell in the __Build and Run__ section below the code to compile and execute the code.

In [41]:
%%writefile lab/sub_group_votes.cpp
//==============================================================
// Copyright © Intel Corporation
//
// SPDX-License-Identifier: MIT
// =============================================================
#include <sycl/sycl.hpp>
using namespace sycl;

static constexpr size_t N = 32; // global size
static constexpr size_t B = 16; // work-group size

int main() {
  queue q;
  std::cout << "Device : " << q.get_device().get_info<info::device::name>() << "\n";

  //# initialize input and output array using usm
  auto input = malloc_shared<int>(N, q);
  auto all = malloc_shared<int>(N, q);
  auto any = malloc_shared<int>(N, q);
  auto none = malloc_shared<int>(N, q);
    
  //# initialize values for input array  
  for(int i=0; i<N; i++) { if (i< 10) input[i] = 0; else input[i] = i; }
  std::cout << "input:\n";
  for(int i=0; i<N; i++) std::cout << input[i] << " "; std::cout << "\n";  

  //# use parallel_for and sub_groups
  q.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item)[[intel::reqd_sub_group_size(32)]] {
    auto sg = item.get_sub_group();
    auto i = item.get_global_id(0);

    //# write items with vote functions
    all[i] = all_of_group(sg, input[i]);
    any[i] = any_of_group(sg, input[i]);
    none[i] = none_of_group(sg, input[i]);

  }).wait();

  std::cout << "all_of:\n";
  for(int i=0; i<N; i++) std::cout << all[i] << " "; std::cout << "\n";
  std::cout << "any_of:\n";
  for(int i=0; i<N; i++) std::cout << any[i] << " "; std::cout << "\n";
  std::cout << "none_of:\n";
  for(int i=0; i<N; i++) std::cout << none[i] << " "; std::cout << "\n";
  
  free(input, q);
  free(all, q);
  free(any, q);
  free(none, q);
  return 0;
}

Overwriting lab/sub_group_votes.cpp


#### Build and Run
Select the cell below and click run ▶ to compile and execute the code:

In [42]:
! chmod 755 q; chmod 755 run_sub_group_votes.sh; if [ -x "$(command -v qsub)" ]; then ./q run_sub_group_votes.sh; else ./run_sub_group_votes.sh; fi

## u3eed2aeb1316a4735ff54db21771fda is compiling SYCL_Essentials Module4 -- SYCL Sub Groups - 6 of 7 sub_group_votes.cpp
Device : Intel(R) Data Center GPU Max 1100
input:
0 0 0 0 0 0 0 0 0 0 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 
all_of:
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 
any_of:
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 
none_of:
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 


## Lab Exercise: Sub-Groups

Complete the coding exercise below using Sub-Group concepts:
- The code has an array `data` of size `N=1024` elements initialized
- We will offload kernel task to compute the sum of all items in each sub-group and save in new array `sg_data`
- We will set the sub-group size to `S=32`, which will make the `sg_data` array of size `N/S`
- Create USM shared allocation for `data` and `sg_data`
- Create a nd-range kernel task with fixed sub-group size of `S`
- In the kernel task, compute the sub-group sum using `reduce_over_group` function
- In the kernel task, save each sub_group sum into the `sg_data` array
- On the host, add all elements of `sg_data` to get the final sum.

1. Edit the code cell below by following the steps and then click run ▶ to save the code to a file.
2. Next run ▶ the cell in the __Build and Run__ section below the code to compile and execute the code.

In [120]:
%%writefile lab/sub_group_lab.cpp
//==============================================================
// Copyright © Intel Corporation
//
// SPDX-License-Identifier: MIT
// =============================================================
#include <sycl/sycl.hpp>
using namespace sycl;

static constexpr size_t N = 1024; // global size
static constexpr size_t B = 256;  // work-group size
static constexpr size_t S = 32;  // sub-group size

int main() {
  queue q;
  std::cout << "Device : " << q.get_device().get_info<info::device::name>() << "\n";

  //# allocate USM shared allocation for input data array and sg_data array
  int *data = malloc_shared<int>(N, q);
  int *sg_data = malloc_shared<int>(N/S, q);
    
  //# initialize input data array
  for (int i = 0; i < N; i++) data[i] = i;
  for (int i = 0; i < N; i++) std::cout << data[i] << " ";
  std::cout << "\n\n";

  //# Kernel task to compute sub-group sum and save to sg_data array
    
  //# STEP 1 : set fixed sub_group size of value S in the kernel below

  q.parallel_for(nd_range<1>(N, B), [=](nd_item<1> item)[[intel::reqd_sub_group_size(S)]] {
    auto sg = item.get_sub_group();
    auto i = item.get_global_id(0);

    //# STEP 2: Add all elements in sub_group using sub_group reduce
      
    int reduction = reduce_over_group(sg, data[i], plus<>());

    //# STEP 3 : save each sub-group sum to sg_data array

    // Use sg and item each as needed
    // Return first element from get_local_id() and get_group_id()
    // Fix calculation of index for sg_data
    size_t group_id = item.get_group()[0];
    size_t sub_group_id = sg.get_group_id()[0];
    size_t num_sub_groups = sg.get_group_range()[0];
    size_t sgd_i = (group_id * num_sub_groups) + sub_group_id;
    if (sg.get_local_id()[0] == 0) {
        sg_data[sgd_i] = reduction;
    }
      
  }).wait();

  //# print sg_data array
  for (int i = 0; i < N/S; i++) std::cout << sg_data[i] << " ";
  std::cout << "\n";
    
  //# STEP 4: compute sum of all elements in sg_data array
  int sum = 0;

  for (size_t i = 0; i < N/S; i++) sum += sg_data[i];

  std::cout << "\nSum = " << sum << "\n";
  
  //# free USM allocations
  free(data, q);
  free(sg_data, q);

  return 0;
}

Overwriting lab/sub_group_lab.cpp


#### Build and Run
Select the cell below and click run ▶ to compile and execute the code:

In [121]:
! chmod 755 run_sub_group_lab.sh; if [ -x "$(command -v qsub)" ]; then ./q run_sub_group_lab.sh; else ./run_sub_group_lab.sh; fi

## u3eed2aeb1316a4735ff54db21771fda is compiling SYCL_Essentials Module4 -- SYCL Sub Groups - 7 of 7 sub_group_lab.cpp
Device : Intel(R) Data Center GPU Max 1100
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 

# <span style="color:blue">Lab Questions</span>

Subgroups allow kernel programming that maps executions at low-level hardware and may help in achieving higher levels of performance. Please answer the following questions in your lab report:

1. When mapping the work-items to a GPU, on which hardware unit are all the work-items in a work-group scheduled? Where are all the work-items in a sub-group executed? Where is each work-item in a sub-group executed?

2. An Intel GPU supports subgroups of size 32, what would be the return value of function __get_local_range()__ ?

3. Why subgroup shuffle operations are preferred over an algorithmic implementation (e.g. using a __for__ loop) ?

4. In code example 1.10.1, what results did you get? Please explain why.

5. In code example 1.10.4, what results did you get? Please explain why.

6. __[EECE.5510 Only]__ If you use a subgroup of size 16, what results would you get for code example 1.10.1 and 1.10.4? Please explain your answer. Hint: you could modify the example codes to verify your hypothesis.